In [1]:
# =========================================================
# === PHASE 1: BUILD AND POPULATE DATABASE SCHEMA ===
# =========================================================



import sqlite3
import os

# --- Define the file structure (Colab-Specific Paths) ---
gdrive_root = "/content/drive/My Drive"
pienza_root = os.path.join(gdrive_root, "_Pienza")
database_folder = os.path.join(pienza_root, 'Assets', 'Database')
architecture_folder = os.path.join(pienza_root, 'Opus1_101001', '06_architecture')
db_file = os.path.join(database_folder, 'opus.db')
schema_file = os.path.join(architecture_folder, 'Opus_1_V5.sql')

# --- Safety Check: Ensure database folder exists ---
if not os.path.exists(database_folder):
    os.makedirs(database_folder)
    print(f"Created database folder at: {database_folder}")

# --- Phase 1: Build the Schema ---
print(f"--- Phase 1: Building database '{db_file}' ---")
if os.path.exists(db_file):
    os.remove(db_file)
    print(f"Removed existing database file to ensure a clean build.")

conn = sqlite3.connect(db_file)
try:
    with open(schema_file, 'r') as f:
        schema_script = f.read()
    print("Successfully read schema file.")
except FileNotFoundError:
    print(f"❌ ERROR: The schema file was not found at '{schema_file}'.")
    exit()

conn.executescript(schema_script)
conn.commit()
conn.close()
print("✅ Schema created successfully.")


# --- Phase 2: Populate All Lookup Tables ---
print(f"\n--- Phase 2: Populating all lookup tables in '{db_file}' ---")

# --- Data Definitions for Lookup Tables ---
product_category_data = [(1, 'uberx'), (2, 'comfort'), (3, 'business_comfort'), (4, 'black'), (5, 'uber_planet'), (6, 'uber_pet'), (7, 'artículo')]
offer_action_data = [(1, 'accepted'), (2, 'reject')]
reason_primary_data = [(1, 'dropoff_non_operational'), (2, 'dropoff_proxy'), (3, 'low_profitability'), (4, 'long_pickup_time'), (5, 'dropoff_strategic_mismatch'), (6, 'expected_value_gamble'), (7, 'system_logic_failure')]
heuristic_flag_data = [(1, 'deadhead_risk'), (2, 'long_ride_traffic_risk'), (3, 'obj_end_session'), (4, 'market_anomaly'), (5, 'friday_traffic_risk'), (6, 'dropoff_uncertain'), (7, 'trap'), (8, 'protest_anomaly')]
post_offer_status_data = [(1, 'continue_game'), (2, 'disconnect')]
driver_state_data = [(1, 'looking_for_rides'), (2, 'trip_in_progress')]
outcome_data = [(1, 'completed'), (2, 'rider_canceled'), (3, 'driver_canceled'), (4, 'unfulfilled'), (5, 'system_failure')]
interpolation_quality_data = [(1, 'unanchored'), (2, 'interpolated'), (3, 'extrapolated_end'), (4, 'extrapolated_start'), (5, 'interpolated_stationary')]
record_status_data = [(1, 'valid'), (2, 'invalid_non_offer')]

conn = sqlite3.connect(db_file)
cur = conn.cursor()

# --- Execute INSERTs for Each Table ---
try:
    cur.executemany("INSERT INTO product_category (product_category_id, category_name) VALUES (?, ?)", product_categories_data)
    print(f"-> Populated 'product_category'.")

    cur.executemany("INSERT INTO offer_action (offer_action_id, offer_action_description) VALUES (?, ?)", offer_action_data)
    print(f"-> Populated 'offer_action'.")

    cur.executemany("INSERT INTO reason_primary (reason_primary_id, reason_primary_description) VALUES (?, ?)", reason_primary_data)
    print(f"-> Populated 'reason_primary'.")

    cur.executemany("INSERT INTO heuristic_flag (heuristic_flag_id, heuristic_flag_description) VALUES (?, ?)", heuristic_flag_data)
    print(f"-> Populated 'heuristic_flag'.")

    cur.executemany("INSERT INTO post_offer_status (post_offer_status_id, post_offer_status_description) VALUES (?, ?)", post_offer_status_data)
    print(f"-> Populated 'post_offer_status'.")

    cur.executemany("INSERT INTO driver_state_at_request (driver_state_id, driver_state_description) VALUES (?, ?)", driver_state_data)
    print(f"-> Populated 'driver_state_at_request'.")

    cur.executemany("INSERT INTO outcome (outcome_id, outcome_description) VALUES (?, ?)", outcome_data)
    print(f"-> Populated 'outcome'.")

    cur.executemany("INSERT INTO interpolation_quality (interpolation_quality_id, interpolation_quality_description) VALUES (?, ?)", interpolation_quality_data)
    print(f"-> Populated 'interpolation_quality'.")

    cur.executemany("INSERT INTO record_status (record_status_id, record_status_description) VALUES (?, ?)", record_status_data)
    print(f"-> Populated 'record_status'.")

    conn.commit()
    print("\n✅ SUCCESS! All lookup tables have been populated.")

except sqlite3.Error as e:
    print(f"\n❌ An SQL error occurred: {e}")
    conn.rollback()
    print("   - All changes have been rolled back.")

finally:
    conn.close()

--- Phase 1: Building database '/content/drive/My Drive/_Pienza/Assets/Database/opus.db' ---
Removed existing database file to ensure a clean build.
❌ ERROR: The schema file was not found at '/content/drive/My Drive/_Pienza/Opus1_101001/06_architecture/Opus_1_V5.sql'.


NameError: name 'schema_script' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# extract_gsheets.ipynb - v2 - Robust and Precise

import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from google.colab import drive

# --- 1. SETUP AND AUTHENTICATION ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive.mount('/content/drive')
print("✅ Authentication successful and Google Drive mounted.")

# --- 2. CONFIGURATION ---
RAW_OCR_SHEET_NAME = "raw_Offers"
SHEET_TAB_NAME = "raw_requests_messy"
STAGING_AREA_PATH = "/content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/"

import os
os.makedirs(STAGING_AREA_PATH, exist_ok=True)
print(f"Staging area is ready at: {STAGING_AREA_PATH}")


# --- 3. EXTRACT AND STAGE: raw_requests_messy (ROBUST METHOD) ---
try:
    print(f"\n--- Extracting '{SHEET_TAB_NAME}' from '{RAW_OCR_SHEET_NAME}' ---")

    workbook = gc.open(RAW_OCR_SHEET_NAME)
    sheet = workbook.worksheet(SHEET_TAB_NAME)

    # Get all values from the sheet as a list of lists
    all_values = sheet.get_all_values()

    # Get the data rows (everything from the second row onwards)
    data_rows = all_values[1:]

    # We will ONLY use the first 10 columns (A-J)
    # And we will assign our own, clean headers.
    clean_headers = [
        'image_filename', 'time_taken_raw', 'ride_type_raw', 'price_raw',
        'pickup_details_raw', 'pickup_loc_raw', 'trip_details_raw', 'dest_address_raw',
        'rider_rating_raw', 'special_note_raw'
    ]

    # Create the DataFrame with only the data and columns we want.
    df = pd.DataFrame(data_rows)
    df = df.iloc[:, 0:10] # Select only the first 10 columns (A-J)
    df.columns = clean_headers

    print(f"Successfully extracted {len(df)} rows, using only the first 10 columns.")

    # --- 4. PRE-TRANSFORM: Clean and Split the data ---
    # This is an extra step to make our final Parquet file even better.

    # A. Filter out invalid rows (important to do this early)
    df = df.dropna(subset=['image_filename']).copy()
    df = df[df['image_filename'] != '']
    print(f"Filtered out rows with no image filename. {len(df)} rows remain.")

    # B. Parse combined fields using Regex
    def parse_details(series, time_pattern, dist_pattern):
        # Using .get() on the result of extract is a safe way to handle no matches
        times = series.str.extract(time_pattern, expand=False).str.strip().astype(float, errors='ignore')
        distances = series.str.extract(dist_pattern, expand=False).str.strip().astype(float, errors='ignore')
        return times, distances

    df['time_to_pickup_min'], df['dist_to_pickup_km'] = parse_details(
        df['pickup_details_raw'],
        r'(\d+)\s*min',
        r'\((\d+\.?\d*)\s*km\)'
    )

    df['est_trip_time_min'], df['est_trip_dist_km'] = parse_details(
        df['trip_details_raw'],
        r'(\d+)\s*min',
        r'\((\d+\.?\d*)\s*km\)'
    )
    print("Parsed composite time/distance fields.")

    # C. Clean the price column
    df['price'] = pd.to_numeric(df['price_raw'].replace({r'[\$,]': ''}, regex=True), errors='coerce')
    print("Cleaned price field.")

    # --- 5. STAGE: Save the clean, structured data ---

    # Define the final columns for our clean, staged file.
    # Notice we are keeping the original raw columns for auditing, PLUS the new parsed ones.
    final_columns = [
        'image_filename', 'time_taken_raw', 'ride_type_raw', 'price_raw',
        'pickup_details_raw', 'pickup_loc_raw', 'trip_details_raw', 'dest_address_raw',
        'rider_rating_raw', 'special_note_raw',
        'time_to_pickup_min', 'dist_to_pickup_km',
        'est_trip_time_min', 'est_trip_dist_km', 'price'
    ]

    output_df = df[final_columns]

    output_file = os.path.join(STAGING_AREA_PATH, "staged_raw_ocr.parquet")
    output_df.to_parquet(output_file, index=False)

    print(f"\n✅ SUCCESS! Cleaned and structured data staged to: {output_file}")
    print("\n--- Sample of the final staged data ---")
    print(output_df.head())

except gspread.exceptions.SpreadsheetNotFound:
    print(f"❌ ERROR: Google Sheet '{RAW_OCR_SHEET_NAME}' not found.")
except gspread.exceptions.WorksheetNotFound:
    print(f"❌ ERROR: Worksheet '{SHEET_TAB_NAME}' not found.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Authentication successful and Google Drive mounted.
Staging area is ready at: /content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/

--- Extracting 'raw_requests_messy' from 'raw_Offers' ---
Successfully extracted 4776 rows, using only the first 10 columns.
Filtered out rows with no image filename. 4776 rows remain.
Parsed composite time/distance fields.
Cleaned price field.

✅ SUCCESS! Cleaned and structured data staged to: /content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/staged_raw_ocr.parquet

--- Sample of the final staged data ---
  image_filename time_taken_raw            ride_type_raw price_raw  \
0   IMG_5452.PNG           6:44                    UberX   $204.24   
1   IMG_5457.PNG           6:45  Uber Priority Exclusivo   $173.86   
2   IMG_5459.PNG           6:45                    UberX   $136.53   
3   IMG_5464.PNG     

In [ ]:
# ==============================================================================
# Step 4 (Definitive, v4): Authenticate, Extract, Stage, and VERIFY
# ==============================================================================

import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
import os
import time # Import the time library for waiting

# --- Part A: AUTHENTICATION ---
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Authentication successful.")

# --- Part B: CONFIGURATION ---
GTS_SHEET_NAME = "GTS-4"
TABS_TO_EXCLUDE = ['data', '250918_query', '250918_analysis', '250918_dashboard', 'Sheet12']
STAGING_AREA_PATH = "/content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/"
os.makedirs(STAGING_AREA_PATH, exist_ok=True)
print(f"Staging area is ready at: {STAGING_AREA_PATH}")
output_file = os.path.join(STAGING_AREA_PATH, "staged_gts_events.parquet")


# --- Part C: EXTRACT AND TRANSFORM ---
try:
    print(f"\n--- Opening Google Sheet: '{GTS_SHEET_NAME}' ---")
    # ... (The entire data extraction and DataFrame creation logic is the same as before)
    # ... It successfully creates 'df_gts'
    workbook = gc.open(GTS_SHEET_NAME)
    all_worksheets = workbook.worksheets()
    list_of_dfs = []
    expected_headers = ['serverTimestamp', 'realTimestamp', 'clientTimestamp', 'rideID', 'eventType', 'latitude', 'longitude', 'addressText', 'upfrontFare', 'realizedFare', 'Is Imputed', 'obs', 'source']
    num_expected_columns = len(expected_headers)
    for sheet in all_worksheets:
        tab_name = sheet.title
        if tab_name in TABS_TO_EXCLUDE: continue
        print(f"   - Extracting tab: '{tab_name}'")
        data = sheet.get_all_values()
        temp_df = pd.DataFrame(data[1:]).iloc[:, :num_expected_columns]
        temp_df.columns = expected_headers
        list_of_dfs.append(temp_df)
    df_gts = pd.concat(list_of_dfs, ignore_index=True)
    df_gts.dropna(subset=['rideID'], inplace=True)
    df_gts = df_gts[df_gts['rideID'] != '']
    print(f"\nSuccessfully extracted and combined {len(df_gts)} valid rows.")

    # --- Part D: STAGING (with a file system flush) ---
    print(f"\n--- Staging data to: {output_file} ---")
    df_gts.to_parquet(output_file, index=False)

    # --- THIS IS THE NEW PART: THE VERIFICATION ---
    print("Staging command executed. Now attempting to verify file existence...")

    # Wait for a moment to allow for sync
    time.sleep(5)

    if os.path.exists(output_file):
        print(f"✅✅✅ VERIFICATION SUCCESS! The file has been created and is visible at the path.")
    else:
        print(f"❌❌❌ VERIFICATION FAILED! The file was NOT found at the specified path.")
        print(f"   -> Please double-check the path for typos and check your Google Drive permissions.")

except Exception as e:
    print(f"❌ An error occurred during the process: {e}")

✅ Authentication successful.
Staging area is ready at: /content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/

--- Opening Google Sheet: 'GTS-4' ---
   - Extracting tab: '250918'

Successfully extracted and combined 1822 valid rows.

--- Staging data to: /content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/staged_gts_events.parquet ---
Staging command executed. Now attempting to verify file existence...
✅✅✅ VERIFICATION SUCCESS! The file has been created and is visible at the path.


In [ ]:
# ==============================================================================
# Step 0: Mount Google Drive
# ==============================================================================
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
    print("✅ Google Drive mounted successfully.")
except Exception as e:
    print(f"❌ Error mounting Google Drive: {e}")
    exit()

# ==============================================================================
# Step 1: Import Libraries and Configure Paths
# ==============================================================================
import pandas as pd
import sqlite3
import os
from tqdm import tqdm

print("\n--- Colab ETL Script: Load Raw OCR Data ---")

# Define file paths using Colab's directory structure
gdrive_root = "/content/drive/My Drive"
pienza_root = os.path.join(gdrive_root, "_Pienza")
assets_root = os.path.join(pienza_root, 'Assets', 'Database')
staging_area_path = os.path.join(assets_root, '02_Staging_Data')
source_file = os.path.join(staging_area_path, 'staged_raw_ocr.parquet')
db_file = os.path.join(assets_root, 'opus.db')

print(f"Target Database: {db_file}")
print(f"Source Parquet File: {source_file}")

# ==============================================================================
# Step 2: EXTRACT
# ==============================================================================
print("\n--- [E] EXTRACTING staged data from Parquet file ---")
try:
    df_staged = pd.read_parquet(source_file)
    print(f"✅ Success: Extracted {len(df_staged)} rows from source.")
except FileNotFoundError:
    print(f"❌ ERROR: Source file not found at '{source_file}'.")
    print("   -> Please ensure the staging notebook has run successfully and the file exists.")
    exit()
except Exception as e:
    print(f"❌ An unexpected error occurred during extraction: {e}")
    exit()

# ==============================================================================
# Step 3: TRANSFORM (Align Schema)
# ==============================================================================
print("\n--- [T] TRANSFORMING data to match the 'raw_offers_ocr' database schema ---")
try:
    df_to_load = pd.DataFrame()

    # This block maps the columns from the Parquet file to the final database column names
    df_to_load['image_filename'] = df_staged['image_filename']
    df_to_load['time_taken'] = df_staged['time_taken_raw']
    df_to_load['ride_type'] = df_staged['ride_type_raw']
    df_to_load['upfront_fare'] = df_staged['price'] # Use the clean numerical column
    df_to_load['time_to_pickup_min'] = df_staged['time_to_pickup_min']
    df_to_load['dist_to_pickup_km'] = df_staged['dist_to_pickup_km']
    df_to_load['pickup_address'] = df_staged['pickup_loc_raw']
    df_to_load['est_trip_time_min'] = df_staged['est_trip_time_min']
    df_to_load['est_trip_dist_km'] = df_staged['est_trip_dist_km']
    df_to_load['dropoff_address'] = df_staged['dest_address_raw']
    df_to_load['rider_rating'] = df_staged['rider_rating_raw']
    df_to_load['special_note'] = df_staged['special_note_raw']

    print("✅ Success: Schema alignment and column renaming complete.")
except KeyError as e:
    print(f"❌ ERROR: A required column was not found in the source Parquet file: {e}")
    exit()
except Exception as e:
    print(f"❌ An unexpected error occurred during transformation: {e}")
    exit()

# ==============================================================================
# Step 4: LOAD
# ==============================================================================
print(f"\n--- [L] LOADING data into the 'raw_offers_ocr' table ---")
table_name = 'raw_offers_ocr'
conn = sqlite3.connect(db_file)

try:
    # Use 'replace' for development to ensure a clean table on every run
    # This will DROP the table if it exists and create a new one.
    print(f"Writing {len(df_to_load)} rows to table '{table_name}'. This may take a moment...")

    df_to_load.to_sql(
        table_name,
        conn,
        if_exists='replace',
        index=False # Do not write the Pandas index as a column
    )

    print(f"✅ Success: Load operation complete for '{table_name}'.")
except sqlite3.OperationalError as e:
    print(f"❌ An SQL Operational Error occurred. This often means a schema mismatch.")
    print(f"   -> Details: {e}")
except Exception as e:
    print(f"❌ An unexpected error occurred during the load operation: {e}")
finally:
    if conn:
        conn.close()
        print("Database connection closed.")

# ==============================================================================
# Step 5: VALIDATE
# ==============================================================================
print("\n--- VALIDATING the load operation ---")
conn = sqlite3.connect(db_file)
try:
    count_validate_df = pd.read_sql_query(f"SELECT COUNT(*) as count FROM {table_name}", conn)
    total_rows = count_validate_df['count'][0]

    print(f"Total records in '{table_name}' table: {total_rows}")

    if total_rows == len(df_to_load):
        print("✅ Row count matches source DataFrame. Validation successful.")
        print("\n--- Sample of 5 records from the live database ---")
        df_validate = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5", conn)
        # Using .to_string() for better formatting of wide tables in Colab
        print(df_validate.to_string())
    else:
        print(f"❌ VALIDATION FAILED: Row count mismatch! Expected {len(df_to_load)}, found {total_rows}.")

except Exception as e:
    print(f"❌ An error occurred during validation: {e}")
finally:
    if conn:
        conn.close()
        print("Database connection for validation closed.")

❌ Error mounting Google Drive: Mountpoint must not already contain files

--- Colab ETL Script: Load Raw OCR Data ---
Target Database: /content/drive/My Drive/_Pienza/Assets/Database/opus.db
Source Parquet File: /content/drive/My Drive/_Pienza/Assets/Database/02_Staging_Data/staged_raw_ocr.parquet

--- [E] EXTRACTING staged data from Parquet file ---
✅ Success: Extracted 4776 rows from source.

--- [T] TRANSFORMING data to match the 'raw_offers_ocr' database schema ---
✅ Success: Schema alignment and column renaming complete.

--- [L] LOADING data into the 'raw_offers_ocr' table ---
Writing 4776 rows to table 'raw_offers_ocr'. This may take a moment...
✅ Success: Load operation complete for 'raw_offers_ocr'.
Database connection closed.

--- VALIDATING the load operation ---
Total records in 'raw_offers_ocr' table: 4776
✅ Row count matches source DataFrame. Validation successful.

--- Sample of 5 records from the live database ---
  image_filename time_taken                ride_type  up

In [ ]:
import pandas as pd
import sqlite3
import os

print("--- Colab ETL Script: Load GTS Trip Events ---")

# ==============================================================================
# Step 1: Configuration
# ==============================================================================
# Define paths using Colab's directory structure
gdrive_root = "/content/drive/My Drive"
pienza_root = os.path.join(gdrive_root, "_Pienza")
assets_root = os.path.join(pienza_root, 'Assets', 'Database')
staging_area_path = os.path.join(assets_root, '02_Staging_Data')
db_file = os.path.join(assets_root, 'opus.db')
source_file = os.path.join(staging_area_path, 'staged_gts_events.parquet')

df_gts = None # Initialize variable to be safe

# ==============================================================================
# Step 2: EXTRACT
# ==============================================================================
print("\n--- [E] EXTRACTING data from all sources ---")
try:
    # A. Extract the staged GTS data from the Parquet file
    df_gts = pd.read_parquet(source_file)
    print(f"✅ Success: Extracted {len(df_gts)} event rows from staged Parquet file.")

    # B. Extract key tables from our database for enrichment
    conn = sqlite3.connect(db_file)
    # CRITICAL ASSUMPTION: 'rideID' from GTS matches 'obs_id' in our final polished 'offers' table
    df_offers_keys = pd.read_sql_query("SELECT offer_id, session_id, obs_id FROM offers", conn)
    print(f"✅ Success: Extracted {len(df_offers_keys)} key rows from 'offers' table.")

    df_event_types = pd.read_sql_query("SELECT event_type_id, event_name FROM event_types", conn)
    print(f"✅ Success: Extracted {len(df_event_types)} rows from 'event_types' lookup table.")

except FileNotFoundError:
    print(f"❌ ERROR: Source file not found at '{source_file}'. Please ensure the staging notebook has been run successfully.")
except Exception as e:
    print(f"❌ ERROR during extraction: {e}")
finally:
    if conn:
        conn.close()

# ==============================================================================
# Step 3: TRANSFORM (Only if Extraction was successful)
# ==============================================================================
if df_gts is not None:
    print("\n--- [T] TRANSFORMING and enriching data ---")

    # A. Data Cleaning and Type Conversion
    df_gts['event_timestamp'] = pd.to_datetime(df_gts['realTimestamp'], errors='coerce')
    df_gts['event_timestamp'].fillna(pd.to_datetime(df_gts['serverTimestamp'], errors='coerce'), inplace=True)
    df_gts['latitude'] = pd.to_numeric(df_gts['latitude'], errors='coerce')
    df_gts['longitude'] = pd.to_numeric(df_gts['longitude'], errors='coerce')
    df_gts['upfront_fare'] = pd.to_numeric(df_gts['upfrontFare'], errors='coerce')
    df_gts['realized_fare'] = pd.to_numeric(df_gts['realizedFare'], errors='coerce')
    print("Cleaned data types.")

    # B. Filtering by Date
    cutoff_date = pd.to_datetime('2025-10-02')
    df_gts_filtered = df_gts[df_gts['event_timestamp'] < cutoff_date].copy()
    print(f"Filtered for events before {cutoff_date}. {len(df_gts_filtered)} rows remain.")

    # C. Enriching with Keys via Merging (the JOINs)
    df_enriched = pd.merge(df_gts_filtered, df_offers_keys, left_on='rideID', right_on='obs_id', how='left')
    print("Enriched with offer_id and session_id.")

    df_enriched = pd.merge(df_enriched, df_event_types, left_on='eventType', right_on='event_name', how='left')
    print("Enriched with event_type_id.")

    # D. Final Schema Alignment
    df_to_load = df_enriched[[
        'event_timestamp', 'latitude', 'longitude', 'addressText',
        'upfront_fare', 'realized_fare', 'source',
        'offer_id', 'session_id', 'event_type_id'
    ]].copy()
    # Rename for perfect DB match
    df_to_load.rename(columns={'addressText': 'address_text'}, inplace=True)
    print("Final schema alignment complete.")

    # ==============================================================================
    # Step 4: LOAD
    # ==============================================================================
    print(f"\n--- [L] LOADING data into 'trip_events' table ---")
    conn = sqlite3.connect(db_file)
    try:
        # Use append since this data should be added, not replace old data.
        df_to_load.to_sql('trip_events', conn, if_exists='append', index=False)
        print(f"✅ SUCCESS! Loaded {len(df_to_load)} records into 'trip_events'.")
    except Exception as e:
        print(f"❌ ERROR during load operation: {e}")
    finally:
        if conn:
            conn.close()

else:
    print("\n--- Halting execution due to error in EXTRACT phase. ---")

--- Colab ETL Script: Load GTS Trip Events ---

--- [E] EXTRACTING data from all sources ---
✅ Success: Extracted 1822 event rows from staged Parquet file.
❌ ERROR during extraction: Execution failed on sql 'SELECT offer_id, session_id, obs_id FROM offers': no such table: offers

--- [T] TRANSFORMING and enriching data ---
Cleaned data types.
Filtered for events before 2025-10-02 00:00:00. 1026 rows remain.


/tmp/ipython-input-1305600340.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_gts['event_timestamp'].fillna(pd.to_datetime(df_gts['serverTimestamp'], errors='coerce'), inplace=True)


NameError: name 'df_offers_keys' is not defined